# Post-Nonlinear Noise Model

In [1]:
import lightning as L
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy import stats

# FIXME: Hack for importing NCP packages
import sys
import os
# Add parent directory to the Python path
sys.path.append(os.path.abspath("../../.."))


def F_X(Z):
    return 0.7 * ((Z**3) / 5 + Z / 2)


def G_X(W):
    return W + (W**3) / 3 + np.tanh(W / 3) / 2


def F_Y(Z):
    return ((Z**3) / 4 + Z) / 3


def G_Y(W):
    return W + np.tanh(W / 3)


def post_nonlinear_model(sample_size, conditional_independent):
    """Sample from post-nonlinear noise model."""
    # Generate Z as vector of i.i.d. standard gaussian components
    Z = stats.norm.rvs(size=(sample_size, 1))

    # Build X and Y from Z using linear, cubic, and tanh functions
    X = G_X(F_X(Z) + np.tanh(stats.norm.rvs(size=(sample_size, 1))))
    if conditional_independent:
        Y = G_Y(F_Y(Z) + stats.norm.rvs(size=(sample_size, 1)))
    else:
        Y = G_Y(F_Y(Z) + X + stats.norm.rvs(size=(sample_size, 1)))

    return X, Y, Z

## Benchmarking post-nonlinear model

In [ ]:
from causallearn.utils.cit import CIT

alpha = 0.05
replications = 1000

type1_errors = 0
for i in range(replications):
    X, Y, Z = post_nonlinear_model(200, conditional_independent=True)

    data = np.hstack((X, Y, Z))

    kci_obj = CIT(data, "kci")
    pValue = kci_obj(0, 1, {2})

    if pValue < alpha:
        type1_errors += 1

type2_errors = 0
for i in range(replications):
    X, Y, Z = post_nonlinear_model(200, conditional_independent=False)

    data = np.hstack((X, Y, Z))

    kci_obj = CIT(data, "kci")
    pValue = kci_obj(0, 1, {2})

    if pValue >= alpha:
        type2_errors += 1

print(f"Type I error: {type1_errors / replications * 100:.2f}%")
print(f"Type II error: {type2_errors / replications * 100:.2f}%")

# Learning Conditional Covariance Operator

## RBF kernel in torch

In [3]:
from sklearn.metrics.pairwise import rbf_kernel as rbf_kernel_sklearn
from scipy.spatial.distance import pdist


def rbf_kernel(X, gamma=None):
    """Naive reimplementation of 'sklearn.metrics.pairwise.rbf_kernel' in torch."""
    if gamma is None:
        gamma = 1.0 / X.shape[1]

    K = torch.sum(X**2, dim=1) - 2*X@X.T + torch.sum(X**2, dim=1).reshape(-1, 1)
    K = torch.exp(-gamma * K)
    return K

# Sanity check
Z = torch.arange(1, 10).reshape(3,3)

print(rbf_kernel(Z, gamma=1).numpy())
print(rbf_kernel_sklearn(Z, gamma=1))

[[1.0000000e+00 1.8795289e-12 0.0000000e+00]
 [1.8795289e-12 1.0000000e+00 1.8795289e-12]
 [0.0000000e+00 1.8795289e-12 1.0000000e+00]]
[[1.00000000e+00 1.87952882e-12 1.24794646e-47]
 [1.87952882e-12 1.00000000e+00 1.87952882e-12]
 [1.24794646e-47 1.87952882e-12 1.00000000e+00]]


## Hybrid NCP-kernel loss

In [ ]:
from NCP.nn.functional import cme_score_opti, cme_score_cov


class CMELoss_Hybrid_Kernel_NCP():
    """Hybrid Conditional Mean Embedding Loss with characteristic kernel on Z."""
    def __init__(self, gamma, dim_x):
        # TODO: Add option to pass other kernel
        self.gamma = gamma
        self.dim_x = dim_x

        # FIXME: Remove
        self.i = 0

    def __call__(self, X, Y, NCP):
        U = NCP.U(X)
        V = NCP.V(Y)

        Z = X[:, self.dim_x:]

        # FIXME: Ugly hack .to("cuda")
        # FIXME: Speed-up: pass gram matrix by reference
        gram_matrix_Z = rbf_kernel(Z, gamma=1.0)

        reg_lambda = 1e-5
        gram_matrix_reg_Z = gram_matrix_Z + reg_lambda * Z.shape[0] * torch.eye(gram_matrix_Z.shape[0], device="cuda")

        L = torch.trace(U.T @ torch.linalg.lstsq(gram_matrix_reg_Z, NCP.S(V)).solution)
        # L = torch.trace(U.T @ NCP.S(V))
        L_NCP = cme_score_cov(X, Y, NCP, self.gamma)

        # FIXME: Remove
        if self.i % 50 == 0:
            print(f"\n\nL={L:.4f}")
            print(f"NCP={L_NCP:.4f}\n")
        self.i += 1

        return 2*L + L_NCP


In [17]:
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from NCP.model import NCPModule, NCPOperator
from NCP.nn.layers import MLP, SingularLayer
from NCP.nn.losses import CMELoss
from NCP.utils import FastTensorDataLoader, from_np
from sklearn.preprocessing import StandardScaler
from torch.nn import Module
from torch.optim import Adam


# TODO: Ugly, remove
class NCPModelCheckpoint(ModelCheckpoint):
    def on_save_checkpoint(self, trainer, pl_module, checkpoint):
        X, Y = trainer.model.batch
        trainer.model.model._compute_conditional_independence_statistic(X, Y)
        checkpoint["state_dict"] = trainer.model.state_dict()


def unbind_namespace(dict):
    """Remove first namespace from dictionary keys.

    Main use case: load state dict and remove "model" prefix.
    e.g., 'model._mean_Ux' => '_mean_Ux'.
    """
    return {".".join(k.split(".")[1:]): v for (k, v) in dict.items()}


def make_ncp(dim_x, dim_y, latent_dim):
    MLP_kwargs_U = {
        "input_shape": dim_x,
        "output_shape": latent_dim,
        "n_hidden": 2,
        "layer_size": 32,
        "dropout": 0,
        "iterative_whitening": False,
        "activation": torch.nn.GELU,
    }

    MLP_kwargs_V = {
        "input_shape": dim_y,
        "output_shape": latent_dim,
        "n_hidden": 2,
        "layer_size": 32,
        "dropout": 0,
        "iterative_whitening": False,
        "activation": torch.nn.GELU,
    }

    return (
        NCPOperator(
            U_operator=MLP,
            V_operator=MLP,
            U_operator_kwargs=MLP_kwargs_U,
            V_operator_kwargs=MLP_kwargs_V,
        ),
        MLP_kwargs_U,
        MLP_kwargs_V,
    )

def experiment(conditional_independent,
               sample_size,
                dim_x,
                dim_y,
                dim_z,
                latent_dim=1):

    # Building data model
    # TODO: Use 'dim_x', 'dim_y', and 'dim_z'
    X, Y, Z = post_nonlinear_model(sample_size=sample_size,
                                   conditional_independent=conditional_independent)
    XZ = np.hstack([X, Z])
    YZ = np.hstack([Y, Z])

    # Processing data before passing it to lightning module
    ntrain, nval = (int(0.9 * XZ.shape[0]), int(0.1 * XZ.shape[0]))

    XZ_train, YZ_train = XZ[:ntrain], YZ[:ntrain]
    XZ_val, YZ_val = XZ[ntrain : ntrain + nval], YZ[ntrain : ntrain + nval]

    xz_scaler = StandardScaler()
    yz_scaler = StandardScaler()

    XZ_train = xz_scaler.fit_transform(XZ_train)
    YZ_train = yz_scaler.fit_transform(YZ_train)

    XZ_val = xz_scaler.transform(XZ_val)
    YZ_val = yz_scaler.transform(YZ_val)

    XZ_train_torch = from_np(XZ_train)
    YZ_train_torch = from_np(YZ_train)
    XZ_val_torch = from_np(XZ_val)
    YZ_val_torch = from_np(YZ_val)

    train_dl = FastTensorDataLoader(
        XZ_train_torch, YZ_train_torch, batch_size=len(XZ_train_torch), shuffle=False
    )
    val_dl = FastTensorDataLoader(
        XZ_val_torch, YZ_val_torch, batch_size=len(XZ_val_torch), shuffle=False
    )

    # NCP Architecture
    ncp_operator, MLP_kwargs_U, MLP_kwargs_V = make_ncp(dim_x=dim_x+dim_z,
                                                        dim_y=dim_y+dim_z,
                                                        latent_dim=latent_dim)
    model = NCPModule(
        ncp_operator,
        optimizer_fn=Adam,
        optimizer_kwargs={"lr": 1e-3},
        loss_fn=CMELoss_Hybrid_Kernel_NCP,
        loss_kwargs={"gamma": 1e-2, "dim_x": dim_x},
    )

    checkpoint = NCPModelCheckpoint(
        filename="NCP",
        save_top_k=1,
        monitor="val_loss",
        mode="min",
    )

    trainer = L.Trainer(
        accelerator="gpu",
        precision="bf16-mixed",
        num_sanity_val_steps=0,
        max_epochs=5000,
        enable_model_summary=False,
        log_every_n_steps=1,
        callbacks=[
            # TQDMProgressBar(leave=False),
            EarlyStopping(monitor="val_loss", mode="min", patience=50),
            checkpoint,
        ],
    )

    # Trains here
    trainer.fit(
        model,
        train_dataloaders=train_dl,
        val_dataloaders=val_dl,
    )

    ncp_state = unbind_namespace(
        torch.load(
            f"{checkpoint.dirpath}/NCP.ckpt",
            weights_only=False,
            map_location=torch.device("cpu"),  # can be removed if using GPU
        )["state_dict"]
    )
    keys_to_pop = [
        "_mean_Ux",
        "_mean_Vy",
        "_sqrt_cov_X_inv",
        "_sqrt_cov_Y_inv",
        "_sing_val",
        "_sing_vec_l",
        "_sing_vec_r",
    ]
    popped_items = {k: ncp_state.pop(k) for k in keys_to_pop}
    ncp = NCPOperator(
        U_operator=MLP,
        V_operator=MLP,
        U_operator_kwargs=MLP_kwargs_U,
        V_operator_kwargs=MLP_kwargs_V,
    )
    ncp.load_state_dict(ncp_state, strict=False)
    for stat, value in popped_items.items():
        ncp.__setattr__(stat, value)

    sing_val = ncp.state_dict()["_sing_val"]
    # Effective rank
    # effective_rank = sing_val.sum().item() / sing_val.max().item()
    return sing_val.max().item()

In [18]:
import logging
from itertools import product

param_grid = {
    "sample_size": [200, 400],
    # "sample_size": np.arange(200, 700, 100),
    # "sample_size": np.arange(200, 20000+200, 200),
    "latent_dim": [1],
    # "latent_dim": [1, 2],
    # "latent_dim": [1, 2, 3, 4, 5],
}

logging.getLogger("pytorch_lightning").setLevel(logging.CRITICAL)

# H0

In [19]:
result = dict()
for sample_size, latent_dim in product(*param_grid.values()):
    result[(sample_size, latent_dim)] = experiment(
        conditional_independent=True,
        sample_size=sample_size,
        dim_x=1,
        dim_y=1,
        dim_z=1,
        latent_dim=latent_dim,
    )

result_h0 = pd.DataFrame(
    data=[(ss, lt, n) for (ss, lt), n in result.items()],
    columns=["sample_size", "latent_dim", "norm"],
)
result_h0
# sns.lineplot(result, x="latent_dim", y="norm", hue="sample_size")

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s] 

/home/afrohlich/NCP/NCP/nn/functional.py:9: UserWarning: Tensor.T is deprecated on 0-D tensors. This function is the identity in these cases. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647406761/work/aten/src/ATen/native/TensorShape.cpp:3691.)
  Cp = 0.5*(C + C.T)




L=-7.1562
NCP=0.0223

Epoch 25:   0%|          | 0/1 [00:00<?, ?it/s, v_num=354, val_loss=-13.6, train_loss=-195.]        

L=-213.0000
NCP=0.1687

Epoch 50:   0%|          | 0/1 [00:00<?, ?it/s, v_num=354, val_loss=104.0, train_loss=-1.71e+3]        

L=-1856.0000
NCP=3.5087

Epoch 75:   0%|          | 0/1 [00:00<?, ?it/s, v_num=354, val_loss=1.98e+3, train_loss=-1.04e+4]        

L=-11008.0000
NCP=88.1316

Epoch 77: 100%|██████████| 1/1 [00:00<00:00, 10.95it/s, v_num=354, val_loss=2.56e+3, train_loss=-1.26e+4]


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s] 

L=1.0469
NCP=0.0201

Epoch 25:   0%|          | 0/1 [00:00<?, ?it/s, v_num=355, val_loss=0.144, train_loss=-6.30]           

L=-6.0938
NCP=0.0423

Epoch 50:   0%|          | 0/1 [00:00<?, ?it/s, v_num=355, val_loss=0.296, train_loss=-33.8]         

L=-37.0000
NCP=0.1674

Epoch 64: 100%|██████████| 1/1 [00:00<00:00,  9.38it/s, v_num=355, val_loss=7.260, train_loss=-278.] 


,sample_size,latent_dim,norm
0,200,1,0.270691
1,400,1,0.300402


# H1

In [ ]:
result = dict()
for sample_size, latent_dim in product(*param_grid.values()):
    result[(sample_size, latent_dim)] = experiment(
        conditional_independent=False,
        sample_size=sample_size,
        dim_x=1,
        dim_y=1,
        dim_z=1,
        latent_dim=latent_dim,
    )

result_h1 = pd.DataFrame(
    data=[(ss, lt, n) for (ss, lt), n in result.items()],
    columns=["sample_size", "latent_dim", "norm"],
)
result_h1
# sns.lineplot(result, x="latent_dim", y="norm", hue="sample_size")